In [1]:
import numpy as np
import time
import matplotlib.pyplot as plt

# ==========================================
# 1. Quadratic Function & Scaled Gradient Descent
# ==========================================
def f_quad(x):
    return x[0]**2 + 4*x[0]*x[1] + 1600*x[1]**2

def grad_f_quad(x):
    return np.array([2*x[0] + 4*x[1], 4*x[0] + 3200*x[1]])

def backtracking_ls_quad(x, d, g, alpha=1.0, rho=0.5, gamma=0.5):
    current_val = f_quad(x)
    slope = np.dot(g, d)
    while f_quad(x + alpha * d) > current_val + gamma * alpha * slope:
        alpha *= rho
        if alpha < 1e-16: break
    return alpha

def solve_gd(x0, tau, rho, use_scaling=False):
    x = np.array(x0, dtype=float)
    k = 0
    max_iter = 10000
    D_diag = np.array([0.5, 1/3200]) # Scaling matrix

    while k < max_iter:
        g = grad_f_quad(x)
        if np.linalg.norm(g) < tau: break
        direction = - (D_diag * g) if use_scaling else -g
        alpha = backtracking_ls_quad(x, direction, g, alpha=1.0, rho=rho, gamma=0.5)
        x = x + alpha * direction
        k += 1
    return x, f_quad(x), k

# ==========================================
# 2. Newton's Method (Sum of Square Roots)
# ==========================================
def q_func(x):
    return np.sqrt(x[0]**2 + 4) + np.sqrt(x[1]**2 + 4)

def grad_q(x):
    return np.array([x[0] / np.sqrt(x[0]**2 + 4), x[1] / np.sqrt(x[1]**2 + 4)])

def hessian_q(x):
    h11 = 4 / (x[0]**2 + 4)**1.5
    h22 = 4 / (x[1]**2 + 4)**1.5
    return np.array([[h11, 0], [0, h22]])

def newton_method(x0, tau, use_line_search=False):
    x = np.array(x0, dtype=float)
    k = 0
    max_iter = 100
    while k < max_iter:
        g = grad_q(x)
        if np.linalg.norm(g) < tau: break
        H_inv = np.linalg.inv(hessian_q(x))
        p = - H_inv @ g

        alpha = 1.0
        if use_line_search:
            while q_func(x + alpha * p) > q_func(x) + 0.5 * alpha * np.dot(g, p):
                alpha *= 0.5
                if alpha < 1e-16: break

        x = x + alpha * p
        k += 1
    return x, q_func(x), k

# ==========================================
# 3. High-Dimensional BFGS (Rosenbrock)
# ==========================================
def f_rosen(x):
    x_i = x[:-1]
    x_next = x[1:]
    return np.sum(4 * (x_i**2 - x_next)**2 + (x_i - 1)**2)

def grad_f_rosen(x):
    n = len(x)
    grad = np.zeros(n)
    x_i = x[:-1]
    x_next = x[1:]
    common = x_i**2 - x_next
    grad[:-1] += 16 * x_i * common + 2 * (x_i - 1)
    grad[1:] += -8 * common
    return grad

def bfgs_solve(n, max_iter=1000, tol=1e-5):
    x = np.zeros(n)
    B = np.eye(n)
    grad = grad_f_rosen(x)
    k = 0
    start_time = time.time()

    while np.linalg.norm(grad) > tol and k < max_iter:
        p = -B.dot(grad)

        # Backtracking Line Search
        alpha = 0.9
        current_f = f_rosen(x)
        grad_dot_p = np.dot(grad, p)
        while f_rosen(x + alpha * p) > current_f + 0.5 * alpha * grad_dot_p:
            alpha *= 0.5
            if alpha < 1e-20: break

        x_new = x + alpha * p
        grad_new = grad_f_rosen(x_new)
        s = x_new - x
        y = grad_new - grad

        sy = np.dot(s, y)
        if sy > 1e-10:
            rho_k = 1.0 / sy
            By = B.dot(y)
            yBy = np.dot(y, By)
            factor = rho_k + rho_k**2 * yBy
            B += factor * np.outer(s, s) - rho_k * (np.outer(By, s) + np.outer(s, By))

        x = x_new
        grad = grad_new
        k += 1

    return time.time() - start_time, k

if __name__ == '__main__':
    print("--- 1. Quadratic GD Scaling ---")
    _, val_gd, iter_gd = solve_gd([1, 4000], 1e-12, 0.5, use_scaling=False)
    _, val_sc, iter_sc = solve_gd([1, 4000], 1e-12, 0.5, use_scaling=True)
    print(f"Standard GD: {iter_gd} iterations")
    print(f"Scaled GD: {iter_sc} iterations")

    print("\n--- 2. Newton's Method ---")
    _, _, iter_fix = newton_method([2, 2], 1e-9, False)
    _, _, iter_ls = newton_method([2, 2], 1e-9, True)
    print(f"Fixed Step Iterations: {iter_fix} (Failed to converge/oscillated)")
    print(f"Backtracking Iterations: {iter_ls}")

    print("\n--- 3. BFGS Scalability Test ---")
    for n in [1000, 2500, 5000]:
        dur, iters = bfgs_solve(n)
        print(f"N={n}: {dur:.2f} seconds, {iters} iterations")

--- 1. Quadratic GD Scaling ---
Standard GD: 10000 iterations
Scaled GD: 17 iterations

--- 2. Newton's Method ---
Fixed Step Iterations: 100 (Failed to converge/oscillated)
Backtracking Iterations: 1

--- 3. BFGS Scalability Test ---
N=1000: 1.10 seconds, 97 iterations
N=2500: 10.05 seconds, 93 iterations
N=5000: 30.79 seconds, 113 iterations
